# Implementing a Policy on the Arm

Run this code to run a trained policy on the arm. Step duration and secondary action scaling have been changed from training values to slow down the real arm.

In [1]:
import os

os.environ["RMW_IMPLEMENTATION"] = "rmw_fastrtps_cpp"
print(os.environ["RMW_IMPLEMENTATION"])

"""
ur3e_policy_deploy.py

Runs the trained UR3e reach policy on real hardware via MyUR3e, using the
overlapping-goal streaming pattern validated in ur3e_myur_streaming_demo.py
(new command every send_period seconds, each scheduled point_duration >
send_period seconds out so the trajectory controller re-splices smoothly),
combined with the policy-inference plumbing from the earlier deployment
draft (obs construction, action scaling, velocity clamp, checkpoint loading).

TWO BUGS FIXED relative to the earlier draft -- confirm these match your
current env cfg before running:
  1. action_scale: was 0.08, now 0.0628 -- must match `scale=` in
     _build_actions() in ur3e_reach_env_cfg.py exactly, or the policy's
     raw actions map to different physical deltas than it was trained on.
  2. step_duration: was 0.1, now 0.08 -- must match decimation * timestep
     in ManagerBasedRlEnvCfg (currently decimation=8, timestep=0.01 ->
     0.08s/step), or the velocity clamp denominator and the trajectory
     controller's `time` argument are both wrong.

If you change either scale or decimation/timestep in the sim cfg later,
update ACTION_SCALE / STEP_DURATION here to match -- these three numbers
(env cfg scale, env cfg decimation*timestep, this script's constants)
must always move together.

DESIGN CHANGE from the earlier draft: uses move_joints (absolute target)
instead of move_joints_r. We already have joint_pos in hand from
build_obs() this step, so computing the absolute target ourselves avoids
move_joints_r's redundant internal blocking read_joints_pos() call --
matters at this send rate, and matches the streaming demo's own stated
preference for move_joints over move_joints_r in high-rate loops.

LOGGING CHANGE: the rollout log is now a plain CSV containing only
timestamp (seconds since loop start), the 6 joint positions, and the 6
joint velocities -- one row per step. See plot_rollout_log.py for a
script that reads this CSV and produces position-vs-time and
velocity-vs-time graphs.
"""

import time

import numpy as np
import torch
import torch.nn as nn
from scipy.spatial.transform import Rotation as R

from myur import MyUR3e

ARM_JOINT_NAMES = [
    "shoulder_pan_joint",
    "shoulder_lift_joint",
    "elbow_joint",
    "wrist_1_joint",
    "wrist_2_joint",
    "wrist_3_joint",
]

# Must match _HOME_QPOS in ur3e_reach_env_cfg.py exactly, same joint order.
HOME_QPOS = [0.0, -1.57, 0.0, -1.57, 0.0, 0.0]

# Empirically validated flange -> gripper-tip correction (see earlier
# calibration). Unchanged from the previous draft.
EE_EXTENSION = 0.1244

# --- The three numbers that must stay in sync with ur3e_reach_env_cfg.py ---
ACTION_SCALE = 0.0628      # must match scale= in _build_actions()
STEP_DURATION = 0.5       # must match decimation * timestep in the sim cfg
SEND_AHEAD_FRACTION = 0.8  # fraction of step_duration to sleep before next
                           # send -- point_duration (STEP_DURATION) stays
                           # longer than send_period (STEP_DURATION *
                           # SEND_AHEAD_FRACTION), giving the overlap
                           # margin the trajectory controller needs to
                           # re-splice smoothly, same ratio validated in
                           # ur3e_myur_streaming_demo.py.

# Real UR3e joint velocity limits (official spec).
REAL_JOINT_VEL_LIMITS = [3.14159, 3.14159, 3.14159, 6.28319, 6.28319, 6.28319]
SAFETY_FACTOR = 0.25  # validated in sim_rollout_to_target.py -- change here
                       # (not by hand-editing CLAMP_LIMITS) if you decide on
                       # a different value after watching the real arm.
CLAMP_LIMITS = [v * SAFETY_FACTOR for v in REAL_JOINT_VEL_LIMITS]

SUCCESS_THRESHOLD = 0.03  # meters, matches success_threshold in commands.py


def get_raw_and_corrected_state(arm: MyUR3e):
    """
    Returns (joint_pos, joint_vel, raw_global_pos, corrected_ee_pos).
    Unchanged from the earlier draft -- see calibration notes there for
    the EE_EXTENSION correction.
    """
    joint_pos = arm.read_joints_pos(degrees=False)
    joint_vel = arm.read_joints_vel(degrees=False)

    x, y, z, rx, ry, rz = arm.read_global_pos()  # degrees=True (default)
    raw_global_pos = [x, y, z, rx, ry, rz]
    ee_position = [x, y, z]

    flange_euler = [rz, ry, rx]
    R_flange = R.from_euler("zyx", flange_euler, degrees=True).as_matrix()
    z_vector = R_flange[:, 2]
    z_vector[2] = -z_vector[2]

    corrected_ee_pos = np.array(ee_position) + EE_EXTENSION * z_vector

    return joint_pos, joint_vel, raw_global_pos, corrected_ee_pos


def build_obs(joint_pos, joint_vel, ee_pos, target_pos):
    """
    Same ordering as ur3e_reach_env_cfg.py's actor_terms: joint_pos_rel,
    joint_vel_rel, target_pos, ee_to_target.

    Takes already-read state as arguments now (rather than re-reading
    inside this function) so the caller can reuse the same read for both
    building the observation and computing this step's absolute target --
    one read per step, not two.
    """
    joint_pos_rel = [p - h for p, h in zip(joint_pos, HOME_QPOS)]
    ee_to_target = [t - e for t, e in zip(target_pos, ee_pos)]
    return list(joint_pos_rel) + list(joint_vel) + list(target_pos) + ee_to_target


def clamp_delta_to_velocity_limits(delta, step_duration, limits):
    """
    Uniform-scale clamp -- identical to the version validated in
    sim_rollout_to_target.py and ClampedRelativeJointPositionAction.
    Scales the WHOLE delta vector by one shared factor so no joint's
    implied velocity exceeds its limit, preserving the coordinated shape
    of the motion.
    """
    implied_vel = [abs(d) / step_duration for d in delta]
    ratios = [v / l for v, l in zip(implied_vel, limits)]
    max_ratio = max(ratios)
    if max_ratio > 1.0:
        scale = 1.0 / max_ratio
        return [d * scale for d in delta], scale
    return list(delta), 1.0


class ObsNormalizer(nn.Module):
    """Rebuilds rsl_rl's EmpiricalNormalization. Unchanged from earlier draft."""

    def __init__(self, dim, eps=1e-2):
        super().__init__()
        self.eps = eps
        self.register_buffer("_mean", torch.zeros(1, dim))
        self.register_buffer("_var", torch.ones(1, dim))
        self.register_buffer("_std", torch.ones(1, dim))
        self.register_buffer("count", torch.tensor(0.0))

    def forward(self, x):
        return (x - self._mean) / (self._std + self.eps)


class NormalizedMLPActor(nn.Module):
    """Rebuilds mjlab/rsl_rl's actor structure. Unchanged from earlier draft."""

    def __init__(self, obs_dim=18, action_dim=6, hidden=(128, 128)):
        super().__init__()
        self.obs_normalizer = ObsNormalizer(obs_dim)
        layers = []
        dims = [obs_dim] + list(hidden) + [action_dim]
        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            if i < len(dims) - 2:
                layers.append(nn.ELU())
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(self.obs_normalizer(x))


def load_actor(checkpoint_path: str, obs_dim: int = 18, action_dim: int = 6,
               hidden=(128, 128)):
    """Unchanged from earlier draft -- see original docstring for the
    strict=False-minus-distribution-keys rationale."""
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state_dict = ckpt["actor_state_dict"]

    actor = NormalizedMLPActor(obs_dim=obs_dim, action_dim=action_dim, hidden=hidden)

    filtered = {k: v for k, v in state_dict.items() if not k.startswith("distribution.")}
    missing, unexpected = actor.load_state_dict(filtered, strict=False)
    assert not unexpected, f"Unexpected keys not consumed: {unexpected}"
    assert not missing, f"Missing keys: {missing}"

    actor.eval()
    return actor


def make_policy_fn(actor):
    def _policy_fn(obs):
        obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            action_tensor = actor(obs_tensor)
        return action_tensor.squeeze(0).numpy().tolist()

    return _policy_fn


def run_policy_loop(
    arm: MyUR3e,
    policy_fn,
    target_pos,
    action_scale: float = ACTION_SCALE,
    step_duration: float = STEP_DURATION,
    send_ahead_fraction: float = SEND_AHEAD_FRACTION,
    num_steps: int = 3,
    log_path: str = None,
    log_period: float = 0.01,
):
    """
    Streams overlapping absolute-position commands toward target_pos,
    matching ur3e_myur_streaming_demo.py's overlap pattern:
      - a new command is dispatched every step_duration*send_ahead_fraction
        seconds (the "send period")
      - each command's point is scheduled step_duration seconds out (the
        "point duration"), which is longer than the send period, so the
        controller re-splices from wherever the arm actually is rather
        than stopping and restarting.

    Stops early once within SUCCESS_THRESHOLD of the target, matching the
    sim task's own success criterion -- but still runs num_steps as a
    hard cap in case the policy never converges (e.g. bad checkpoint,
    unreachable target).

    The rollout log written to log_path is a plain CSV with one row per
    sample: elapsed timestamp (seconds since the loop started), the 6
    joint positions, then the 6 joint velocities. Use plot_rollout_log.py
    to turn this into position-vs-time / velocity-vs-time graphs.

    Logging is decoupled from the policy send rate: one sample is taken
    at the top of each policy step (reusing the state already read for
    obs construction), and then additional samples are taken every
    log_period seconds (default 0.01s) during the wait before the next
    send. This does not change when commands are actually sent to the
    arm -- it only samples more often during the window the loop was
    already going to spend waiting, giving a much finer-grained log for
    plotting than one point per policy step.
    """
    joint_pos_headers = [f"pos_{name}" for name in ARM_JOINT_NAMES]
    joint_vel_headers = [f"vel_{name}" for name in ARM_JOINT_NAMES]
    log_lines = ["timestamp," + ",".join(joint_pos_headers) + "," + ",".join(joint_vel_headers) + "\n"]

    loop_start = time.time()
    prev_joint_pos = None
    prev_t = None

    for step in range(num_steps):
        step_start = time.time()

        joint_pos, joint_vel, _raw_global_pos, ee_pos = get_raw_and_corrected_state(arm)
        obs = build_obs(joint_pos, joint_vel, ee_pos, target_pos)
        action = policy_fn(obs)
        raw_delta = [action_scale * a for a in action]
        # secondary action scaling
        raw_delta = [0.5* a for a in raw_delta]
        print(action, raw_delta)

        clamped_delta, scale_factor = clamp_delta_to_velocity_limits(
            raw_delta, step_duration, CLAMP_LIMITS
        )
        if scale_factor < 1.0:
            print(f"step {step}: velocity clamp engaged, scale={scale_factor:.3f}")

        target_joint_positions = [p + d for p, d in zip(joint_pos, clamped_delta)]

        # move_joints (absolute), not move_joints_r -- we already have
        # joint_pos from get_raw_and_corrected_state above, so computing
        # the absolute target ourselves avoids move_joints_r's redundant
        # internal blocking read. time must be a tuple for single-point
        # calls (see streaming demo docstring re: make_trajectory quirk).
        arm.move_joints(
            target_joint_positions,
            time=(step_duration, step_duration),
            degrees=False,
            wait=False,
            vis_only=False,
        )

        ee_dist = float(np.linalg.norm(np.array(target_pos) - np.array(ee_pos)))

        # Realized velocity from the previous step's actual joint motion,
        # not the commanded delta -- this is the number that matters for
        # confirming the clamp is holding on real hardware, same check
        # recommended for the sim rollout log.
        if prev_joint_pos is not None:
            dt = step_start - prev_t
            realized_vel = [abs(p - pp) / dt for p, pp in zip(joint_pos, prev_joint_pos)]
            over_limit = [v > l * 1.05 for v, l in zip(realized_vel, CLAMP_LIMITS)]  # 5% margin for read jitter
            if any(over_limit):
                print(f"step {step}: WARNING realized velocity exceeds clamp: {realized_vel}")
        prev_joint_pos = joint_pos
        prev_t = step_start

        elapsed_timestamp = step_start - loop_start
        pos_str = ",".join(f"{p:.6f}" for p in joint_pos)
        vel_str = ",".join(f"{v:.6f}" for v in joint_vel)
        log_lines.append(f"{elapsed_timestamp:.4f},{pos_str},{vel_str}\n")

        if step % 10 == 0:
            print(f"step {step}: ee_dist={ee_dist:.4f}  scale={scale_factor:.3f}  joint_pos={[round(v, 4) for v in joint_pos]}")

        if ee_dist < SUCCESS_THRESHOLD:
            print(f"step {step}: reached target (ee_dist={ee_dist:.4f} < {SUCCESS_THRESHOLD}), stopping early.")
            break

        send_period = step_duration * send_ahead_fraction
        elapsed = time.time() - step_start
        if elapsed >= send_period:
            print(f"step {step}: WARNING loop took {elapsed:.4f}s, longer than send period "
                  f"{send_period:.4f}s -- real send rate is falling behind.")
        else:
            # Instead of one long sleep, wake up every log_period seconds
            # (default 0.01s) to sample and log joint state. This is
            # purely a logging change -- the next command is still sent
            # at the same time, right when send_period elapses.
            while True:
                now = time.time()
                remaining = send_period - (now - step_start)
                if remaining <= 0:
                    break
                time.sleep(min(log_period, remaining))

                sample_t = time.time()
                sample_pos = arm.read_joints_pos(degrees=False)
                sample_vel = arm.read_joints_vel(degrees=False)
                ts = sample_t - loop_start
                pos_str = ",".join(f"{p:.6f}" for p in sample_pos)
                vel_str = ",".join(f"{v:.6f}" for v in sample_vel)
                log_lines.append(f"{ts:.4f},{pos_str},{vel_str}\n")

    if log_path:
        with open(log_path, "w") as f:
            f.writelines(log_lines)
        print(f"Logged {len(log_lines) - 1} samples to {log_path}")


def wait_for_last_goal_then_stop(arm: MyUR3e, point_duration: float):
    """
    Same pattern as the streaming demo: wait for the in-flight goal to
    report done before calling stop(), avoiding the "generator already
    executing" race between the background spin thread and a stop()
    call from the main thread.
    """
    timeout = point_duration + 1
    wait_start = time.time()
    while not arm.done and (time.time() - wait_start) < timeout:
        time.sleep(0.005)

    if not arm.done:
        arm.print_info(
            "Warning: last streamed goal did not report done before timeout; "
            "calling stop() anyway."
        )
    arm.stop()


def main():
    checkpoint_path = "2026-07-23_14-02-34.pt"  # update to your actual checkpoint
    target_pos = [0.3, 0, 0.3]        # world-frame target, meters -- match
                                          # whatever you validated in
                                          # sim_rollout_to_target.py first

    actor = load_actor(checkpoint_path)
    policy_fn = make_policy_fn(actor)

    arm = MyUR3e()
    arm.set_debug_level(False)

    try:
        run_policy_loop(
            arm,
            policy_fn,
            target_pos,
            num_steps=50,
            log_path="real_rollout_log_3_0_3.txt",
        )
    except KeyboardInterrupt:
        print("Interrupted -- stopping arm.")
    finally:
        #wait_for_last_goal_then_stop(arm, point_duration=STEP_DURATION)
        del arm  # calls rclpy.shutdown() internally


if __name__ == "__main__":
    main()
    print("done")

rmw_fastrtps_cpp
[-0.32506516575813293, 2.4231839179992676, 0.9487488269805908, 1.7742635011672974, -2.4599618911743164, 0.7586818933486938] [-0.010207046204805373, 0.076087975025177, 0.02979071316719055, 0.05571187393665313, -0.07724280338287352, 0.023822611451148983]
step 0: ee_dist=0.6343  scale=1.000  joint_pos=[0.0665, -1.6088, -0.0562, -1.5708, -0.0, -0.0]
step 0: WARNING loop took 0.7981s, longer than send period 0.4000s -- real send rate is falling behind.
[-0.32506635785102844, 2.4231488704681396, 0.9487179517745972, 1.7742295265197754, -2.459960460662842, 0.7586673498153687] [-0.010207083636522292, 0.07608687453269958, 0.029789743685722347, 0.05571080713272094, -0.07724275846481322, 0.023822154784202573]


[INFO] [1785178001.674997092] [remote_control_client]: joints: [0.05627357549071312, -1.5327192511954248, -0.02641242491602898, -1.5150618655259598, -0.07726456304659061, 0.023814680065019782]
[INFO] [1785178001.675459531] [remote_control_client]: time: (0.5, 0.5)
[INFO] [1785178001.675729082] [remote_control_client]: Goal #1: Executing
[INFO] [1785178001.680846943] [remote_control_client]: joints: [0.056285473889112474, -1.5327394251742303, -0.02639835540056229, -1.515056042034411, -0.0772251777175872, 0.023818038095338997]
[INFO] [1785178001.681098285] [remote_control_client]: time: (0.5, 0.5)
[INFO] [1785178001.681375543] [remote_control_client]: Goal #2: Executing
[INFO] [1785178001.682180764] [remote_control_client]: Goal #1: Replaced with Goal #2


[-0.38262704014778137, 2.41833233833313, 1.2310572862625122, 2.2571005821228027, -2.9225800037384033, 0.6937248706817627] [-0.012014489060640334, 0.07593563542366027, 0.03865519878864288, 0.070872958278656, -0.09176901211738586, 0.021782960939407348]


[INFO] [1785178002.080342549] [remote_control_client]: joints: [0.04737186115384102, -1.4797446565693995, 0.0031489354133605943, -1.4610049741259594, -0.14565668135751897, 0.03838887085318565]
[INFO] [1785178002.081384224] [remote_control_client]: time: (0.5, 0.5)
[INFO] [1785178002.082142764] [remote_control_client]: Goal #3: Executing
[INFO] [1785178002.092132336] [remote_control_client]: Goal #2: Replaced with Goal #3


[-0.47659745812416077, 2.290024757385254, 1.3350131511688232, 2.2579734325408936, -3.031567335128784, 0.6416159868240356] [-0.014965160185098646, 0.07190677738189696, 0.04191941294670105, 0.07090036578178405, -0.09519121432304381, 0.020146741986274717]


...

[INFO] [1785178011.700958359] [remote_control_client]: joints: [-0.3664666756378573, -1.2790455549529571, 0.525127383123533, -0.7250009625724335, -1.5418641819344918, 0.1505022853240371]
[INFO] [1785178011.701784187] [remote_control_client]: time: (0.5, 0.5)
[INFO] [1785178011.702717862] [remote_control_client]: Goal #27: Executing
[INFO] [1785178011.718007720] [remote_control_client]: Goal #26: Replaced with Goal #27


[-0.4142525792121887, -0.19533643126487732, 0.33612656593322754, 0.23731541633605957, -1.0257267951965332, -0.04719097167253494] [-0.013007530987262724, -0.006133563941717148, 0.010554374170303344, 0.00745170407295227, -0.03220782136917114, -0.001481796510517597]


[INFO] [1785178012.103526301] [remote_control_client]: joints: [-0.37483997485508136, -1.2828081784955043, 0.532087688552038, -0.7206994546226044, -1.5625545169459742, 0.14948007940500974]
[INFO] [1785178012.104875946] [remote_control_client]: time: (0.5, 0.5)
[INFO] [1785178012.107902001] [remote_control_client]: Goal #27: Replaced with Goal #28
[INFO] [1785178012.108449677] [remote_control_client]: Goal #28: Executing


[-0.31993237137794495, -0.14523214101791382, 0.28825098276138306, 0.15582028031349182, -0.8177602291107178, -0.042913682758808136] [-0.01004587646126747, -0.004560289227962494, 0.009051080858707428, 0.0048927568018436425, -0.025677671194076535, -0.0013474896386265753]
step 28: reached target (ee_dist=0.0288 < 0.03), stopping early.
Logged 1008 samples to real_rollout_log_3_0_3.txt
done


[INFO] [1785178012.502918812] [remote_control_client]: joints: [-0.3809854332136076, -1.2854746715715906, 0.5379644039379519, -0.7179737348786374, -1.57861786500086, 0.14861174674481153]
[INFO] [1785178012.504076396] [remote_control_client]: time: (0.5, 0.5)
[INFO] [1785178012.504873951] [remote_control_client]: Goal #29: Executing
[INFO] [1785178012.522168031] [remote_control_client]: Goal #28: Replaced with Goal #29
[INFO] [1785178013.059196789] [remote_control_client]: Goal #29: Completed
